<a href="https://colab.research.google.com/github/AhmedEssam1996/OCR/blob/main/DMS_Zone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. تثبيت أداة تحويل صفحات الـ PDF إلى صور
!apt-get install -y poppler-utils > /dev/null

# 2. تثبيت مكتبات بايثون (بدون تيسيراكت التقليدي!)
!pip install --quiet langchain langchain-community langchain-huggingface langchain-chroma groq pypdf python-docx pandas openpyxl pdf2image tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/

In [32]:
import os
import base64
import logging
import uuid
from pathlib import Path
from typing import List, Optional, Any

import pandas as pd
from docx import Document as DocxDocument
from pdf2image import convert_from_path
from pypdf import PdfReader

from groq import Groq
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s]: %(message)s")
logger = logging.getLogger("GroqHybridRAG")


class GroqFileProcessor:
    """
    Handles document processing across multiple formats and utilizes
    Llama-4-Scout (Vision) for high-fidelity OCR operations.
    """

    def __init__(self, groq_client: Groq):
        self.client: Groq = groq_client
        self.ocr_model: str = "meta-llama/llama-4-scout-17b-16e-instruct"

    def _call_llama4_vision_ocr(self, image_path: str) -> str:
        """Converts image to Base64 and sends it to Groq for Markdown extraction."""
        logger.info("Sending document to Llama-4-Scout for visual OCR...")

        try:
            with open(image_path, "rb") as img_file:
                base64_image = base64.b64encode(img_file.read()).decode("utf-8")

            response = self.client.chat.completions.create(
                model=self.ocr_model,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": (
                                    "Perform a comprehensive, high-fidelity OCR on this entire document image. "
                                    "CRITICAL: Do NOT skip the top or header sections. You MUST extract all metadata, "
                                    "company names, invoice numbers, dates, tax numbers, and billing details from the top, "
                                    "in addition to the financial tables below. "
                                    "Convert all tables into perfectly aligned clean Markdown tables. "
                                    "Output ONLY the complete markdown result, without any intro, conversational text, or explanation."
                                )
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/png;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                temperature=0.1
            )
            return response.choices[0].message.content or ""
        except Exception as e:
            logger.error(f"Error during Llama-4 Vision call: {str(e)}")
            return ""

    def extract_text(self, file_path: str) -> str:
        """Identifies file extension and routes it to the correct extraction pipeline."""
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found at path: {file_path}")

        ext = Path(file_path).suffix.lower()
        text = ""

        logger.info(f"Processing file with extension: {ext}")

        if ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

            if not text.strip():
                logger.info("Scanned PDF detected. Activating visual OCR engine...")
                images = convert_from_path(file_path)
                for i, img in enumerate(images):
                    temp_path = f"/content/temp_page_{i}.png"
                    try:
                        img.save(temp_path, "PNG")
                        text += self._call_llama4_vision_ocr(temp_path) + "\n"
                    finally:
                        if os.path.exists(temp_path):
                            os.remove(temp_path)

        elif ext in [".png", ".jpg", ".jpeg"]:
            text = self._call_llama4_vision_ocr(file_path)

        elif ext in [".xlsx", ".xls", ".csv"]:
            df = pd.read_csv(file_path) if ext == ".csv" else pd.read_excel(file_path)
            text = df.to_markdown(index=False)

        elif ext == ".docx":
            doc = DocxDocument(file_path)
            text = "\n".join([p.text for p in doc.paragraphs])

        elif ext == ".txt":
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        else:
            raise ValueError(f"File type {ext} is not supported.")

        return text


class UltimateRAGSystem:
    """
    Manages Vector DB initialization, document context isolation,
    and generation of structured, document-mapped streaming responses.
    """

    def __init__(self):
        self.embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

        api_key: Optional[str] = userdata.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("API Key missing. Please add GROQ_API_KEY to your Secrets panel.")

        self.client = Groq(api_key=api_key)
        self.processor = GroqFileProcessor(self.client)
        self.retriever: Optional[Any] = None

    def ingest_file(self, file_path: str) -> None:
        """Parses the document, splits it into chunks, and creates an isolated vector collection."""
        raw_text = self.processor.extract_text(file_path)

        if not raw_text.strip():
            raise ValueError("Failed to extract any text content from the document.")

        chunks = self.text_splitter.split_text(raw_text)
        docs = [Document(page_content=chunk, metadata={"source": file_path}) for chunk in chunks]

        logger.info(f"Document successfully split into {len(docs)} chunks.")

        collection_name = f"doc_{uuid.uuid4().hex}"
        vectorstore = Chroma.from_documents(
            documents=docs,
            embedding=self.embedding_model,
            collection_name=collection_name
        )

        self.retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
        logger.info("Vector DB is ready for the current file context.")

    def ask_stream(self, question: str) -> None:
        """Retrieves context with explicit document mapping and streams structured answers."""
        if not self.retriever:
            raise RuntimeError("No active database found. Run 'ingest_file' before asking questions.")

        retrieved_docs = self.retriever.invoke(question)

        context_entries = []
        for doc in retrieved_docs:
            source_file = os.path.basename(doc.metadata.get("source", "Unknown_Source"))
            context_entries.append(f"Source Document: {source_file}\nContent:\n{doc.page_content}")

        context = "\n\n=========================================\n\n".join(context_entries)

        prompt = f"""You are a strict and highly precise data analyst. Your task is to extract information from the provided context and organize it strictly by the user's question and the specific source document.

**STRICT RULES:**
1. **NO EXTERNAL KNOWLEDGE:** Use ONLY the provided Context. Do not assume or guess or bring details outside this text.
2. **DOCUMENT-BASED GROUPING:** You MUST organize the output by the user's question, and then create a dedicated section or table for each specific Source Document that contains the answer.
3. **STRUCTURE:** Under each document header, present the extracted fields and values inside a clean, perfectly formatted Markdown Table.
4. **NO HALLUCINATION:** If the answer for a specific question or a specific document is not available in the context, output 'المعلومة غير متوفرة' under that specific section.
5. **LANGUAGE:** You MUST translate and output the final answer entirely in Arabic (اللغة العربية).

--- Context ---
{context}

--- User Question ---
{question}

Answer:"""

        print(f"\nAsk: {question}")
        print("Response: \n", end="", flush=True)

        stream = self.client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a strict document QA bot that structures answers by question and document source in Arabic."
                },
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.1,
            stream=True
        )

        for chunk in stream:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)

        print("\n\n" + "="*60)

In [35]:
from IPython.display import clear_output

clear_output(wait=True)

rag = UltimateRAGSystem()

FILE_PATH = "/content/c4283b4abf5bff838cb9e999499ab5439ec90ea2.png"

rag.ingest_file(FILE_PATH)

QUERY = "extract full information"
rag.ask_stream(QUERY)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ask: extract full information
Response: 
### سؤال المستخدم: استخراج المعلومات الكاملة
#### وثيقة المصدر: c4283b4abf5bff838cb9e999499ab5439ec90ea2.png
| الحقل | القيمة |
| --- | --- |
| رقم الفاتورة | 12 |
| تاريخ الإصدار | 2014-04-22 |
| تاريخ الاستحقاق | 2014-05-06 |
| نوع الدفع | تحويل / Transferencia |
| البائع | شركة ACME |
| عنوان البائع | 661 coney island ave, NY 11211 نيويورك, الولايات المتحدة / Estados Unidos |
| رقم التعريف الضريبي / CIF | AAGB860519G32 |
| الفاكس | (212) 446 8571 |
| الهاتف | (212) 446 7001 |
| المدينة | 01234567890123456789012 |
| المشتري | بينتون, جون ب جونيور |
| عنوان المشتري | 6649 N Blue Gum St, 70116 نيو أورليانز, الولايات المتحدة / Estados Unidos |
| رقم التعريف الضريبي / CIF للمشتري | 1AGB860519G32 |
| إجمالي السعر الشبكي | 24.00 |
| مبلغ الضريبة على القيمة المضافة / IVA | 4.80 |
| إجمالي السعر الإجمالي | 28.80 |
| المبلغ المستحق | 28.80 |
| بالكلمات | ثمانية وعشرون دولار وثمانون سنت |
| توقيع البائع | [توقيع البائع / Firma Vendedor] |

